# Notebook 6 — Temporal Models Comparison (LSTM Autoencoder vs GRU Autoencoder)

## Purpose
This notebook compares two temporal models for the Diagnostic Agent sequence branch:

- **LSTM Autoencoder**
- **GRU Autoencoder**

## Current benchmark setting
- real data only
- current supervised benchmark classes:
  - `RC_CAPACITY_OVERLOAD`
  - `RC_TRANSPORT_DELAY`
- main temporal benchmark window size:
  - **10 samples**

This notebook is designed to produce:
- training/validation loss curves
- reconstruction error comparison
- latent-space visualizations
- presentation-ready Plotly figures
- a clear winner for the temporal branch

## Why this notebook comes now

From Notebook 3, the main temporal benchmark uses:
- sequence windows of 5, 10, and 20
- with **10-sample windows** as the main benchmark

From Notebook 4, the tabular supervised branch is already validated, with **Random Forest** selected as the best baseline model after patching.

This notebook now validates the **temporal sequence branch** of the Diagnostic Agent.

In [1]:
import os
import gc
import time
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

VERBOSE = True
SAVE_ARTIFACTS = True
EXPORT_PLOTS = True
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

def log(msg, level="INFO"):
    if VERBOSE:
        print(f"[{level}] {msg}")

def banner(title):
    if VERBOSE:
        print("\n" + "=" * 90)
        print(title)
        print("=" * 90)

def timed_run(name, fn, *args, **kwargs):
    start = time.time()
    log(f"START: {name}")
    result = fn(*args, **kwargs)
    elapsed = time.time() - start
    log(f"END: {name} | elapsed = {elapsed:.2f} sec")
    return result

banner("NOTEBOOK 6 INITIALIZATION")
log("Notebook 6 initialized")


NOTEBOOK 6 INITIALIZATION
[INFO] Notebook 6 initialized


In [2]:
banner("INSTALL / IMPORT VISUALIZATION PACKAGES")

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    import sys
    !{sys.executable} -m pip install -q plotly
    import plotly.express as px
    import plotly.graph_objects as go

try:
    import kaleido  # noqa: F401
    log("kaleido already available")
except Exception:
    log("Installing kaleido for Plotly PNG export...", "WARN")
    import sys
    !{sys.executable} -m pip install -q kaleido
    import kaleido  # noqa: F401
    log("kaleido installed successfully")

def save_plot(fig, name):
    if not EXPORT_PLOTS:
        return
    fig.write_html(f"{name}.html")
    try:
        fig.write_image(f"{name}.png", scale=2)
        log(f"Saved {name}.html and {name}.png")
    except Exception as e:
        log(f"Saved {name}.html | PNG export skipped ({e})", "WARN")


INSTALL / IMPORT VISUALIZATION PACKAGES
[WARN] Installing kaleido for Plotly PNG export...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.7 MB/s eta 0:00:00
[INFO] kaleido installed successfully


## Device compatibility note

Some Kaggle accelerator environments report CUDA as available, but the installed PyTorch build may not support the exact GPU architecture for recurrent layers like LSTM/GRU.

This corrected notebook therefore performs a **real CUDA compatibility test** before selecting the device.
If the test fails, it automatically falls back to **CPU**.

In [3]:
banner("INSTALL / IMPORT MODELING PACKAGES")

# Torch
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except Exception:
    log("torch not found. Installing now...", "WARN")
    import sys
    !{sys.executable} -m pip install -q torch
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

# Sklearn utilities for evaluation/visualization
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import joblib

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

def select_safe_device():
    """Choose CUDA only if it is actually usable for recurrent models, otherwise fall back to CPU."""
    if not torch.cuda.is_available():
        return "cpu", "CUDA not available"

    try:
        # Basic CUDA tensor test
        _ = torch.tensor([1.0], device="cuda")

        # Recurrent-model compatibility test
        test_model = nn.LSTM(input_size=4, hidden_size=4, batch_first=True).to("cuda")
        test_x = torch.randn(2, 3, 4, device="cuda")
        with torch.no_grad():
            _ = test_model(test_x)

        # Cleanup
        del test_model, test_x, _
        torch.cuda.empty_cache()
        return "cuda", "CUDA recurrent compatibility test passed"
    except Exception as e:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return "cpu", f"CUDA unusable for this environment: {e}"

DEVICE, DEVICE_REASON = select_safe_device()
log(f"Using device: {DEVICE}")
log(f"Device decision reason: {DEVICE_REASON}")

# Extra safety note for Kaggle environments with incompatible GPU architecture
if DEVICE == "cpu":
    log("Falling back to CPU is acceptable here because the benchmark dataset is still relatively small.", "WARN")


INSTALL / IMPORT MODELING PACKAGES
[INFO] Using device: cpu
[INFO] Device decision reason: CUDA unusable for this environment: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

[WARN] Falling back to CPU is acceptable here because the benchmark dataset is still relatively small.


## Load Notebook 3 sequence artifacts

We use the main temporal benchmark window:
- **10 samples**

We load:
- `seq_w10_train.npz`
- `seq_w10_val.npz`
- `seq_w10_test.npz`
- metadata parquet files for labels and timestamps

In [4]:
banner("LOAD NOTEBOOK 3 SEQUENCE ARTIFACTS")

WINDOW_SIZE = 10

train_npz = np.load(f"/kaggle/input/datasets/azizamriiiiiiiiii/note6-datas/seq_w{WINDOW_SIZE}_train.npz", allow_pickle=True)
val_npz   = np.load(f"/kaggle/input/datasets/azizamriiiiiiiiii/note6-datas/seq_w{WINDOW_SIZE}_val.npz", allow_pickle=True)
test_npz  = np.load(f"/kaggle/input/datasets/azizamriiiiiiiiii/note6-datas/seq_w{WINDOW_SIZE}_test.npz", allow_pickle=True)

X_train = train_npz["X"]
y_train_raw = train_npz["y"]


X_val = val_npz["X"]
y_val_raw = val_npz["y"]

X_test = test_npz["X"]
y_test_raw = test_npz["y"]

meta_train = pd.read_parquet(f"/kaggle/input/datasets/azizamriiiiiiiiii/note6-datas/seq_w{WINDOW_SIZE}_meta_train.parquet")
meta_val   = pd.read_parquet(f"/kaggle/input/datasets/azizamriiiiiiiiii/note6-datas/seq_w{WINDOW_SIZE}_meta_val.parquet")
meta_test  = pd.read_parquet(f"/kaggle/input/datasets/azizamriiiiiiiiii/note6-datas/seq_w{WINDOW_SIZE}_meta_test.parquet")

log(f"X_train shape = {X_train.shape}")
log(f"X_val shape   = {X_val.shape}")
log(f"X_test shape  = {X_test.shape}")
display(meta_train.head(2))


LOAD NOTEBOOK 3 SEQUENCE ARTIFACTS
[INFO] X_train shape = (354, 10, 16)
[INFO] X_val shape   = (58, 10, 16)
[INFO] X_test shape  = (68, 10, 16)


,group_name,start_timestamp,end_timestamp,label,choice_id,source_file,row_id
0,qos_timeseries_choice_11_20260327.csv,2026-03-27 17:22:08.968812,2026-03-27 20:08:31.306472,RC_CAPACITY_OVERLOAD,11,qos_timeseries_choice_11_20260327.csv,0
1,qos_timeseries_choice_11_20260327.csv,2026-03-27 17:34:37.461783,2026-03-27 20:14:28.418886,RC_CAPACITY_OVERLOAD,11,qos_timeseries_choice_11_20260327.csv,1


## Encode labels and inspect sequence class balance

Even though the autoencoders are trained unsupervised, the labels are useful for:
- latent-space separability analysis
- reconstruction error by class
- presentation figures

In [5]:
banner("ENCODE LABELS AND PLOT CLASS BALANCE")

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)
class_names = list(le.classes_)

log(f"Class names = {class_names}")

balance_df = pd.DataFrame({
    "split": (["train"] * len(y_train_raw)) + (["val"] * len(y_val_raw)) + (["test"] * len(y_test_raw)),
    "label": np.concatenate([y_train_raw, y_val_raw, y_test_raw])
})

balance_plot_df = (
    balance_df.groupby(["split", "label"])
    .size()
    .reset_index(name="rows")
)

fig_balance = px.bar(
    balance_plot_df,
    x="split",
    y="rows",
    color="label",
    barmode="group",
    title=f"Sequence Class Balance by Split (w={WINDOW_SIZE})",
    text="rows"
)
fig_balance.update_layout(template="plotly_white")
fig_balance.show()

save_plot(fig_balance, "nb6_sequence_class_balance")


ENCODE LABELS AND PLOT CLASS BALANCE
[INFO] Class names = [np.str_('RC_CAPACITY_OVERLOAD'), np.str_('RC_TRANSPORT_DELAY')]


[WARN] Saved nb6_sequence_class_balance.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Standardize sequence features

We standardize features using the training split only.

This is important for stable recurrent training.

In [6]:
banner("STANDARDIZE SEQUENCE FEATURES")

n_train, seq_len, n_features = X_train.shape

scaler = StandardScaler()
X_train_2d = X_train.reshape(-1, n_features)
X_val_2d   = X_val.reshape(-1, n_features)
X_test_2d  = X_test.reshape(-1, n_features)

X_train_scaled = scaler.fit_transform(X_train_2d).reshape(X_train.shape).astype("float32")
X_val_scaled   = scaler.transform(X_val_2d).reshape(X_val.shape).astype("float32")
X_test_scaled  = scaler.transform(X_test_2d).reshape(X_test.shape).astype("float32")

log(f"Scaled train shape = {X_train_scaled.shape}")
log(f"Scaled val shape   = {X_val_scaled.shape}")
log(f"Scaled test shape  = {X_test_scaled.shape}")

joblib.dump(scaler, "nb6_sequence_scaler.joblib")
log("Saved nb6_sequence_scaler.joblib")


STANDARDIZE SEQUENCE FEATURES
[INFO] Scaled train shape = (354, 10, 16)
[INFO] Scaled val shape   = (58, 10, 16)
[INFO] Scaled test shape  = (68, 10, 16)
[INFO] Saved nb6_sequence_scaler.joblib


## Dataset and DataLoader setup

We create simple PyTorch datasets for the sequence windows.

In [7]:
banner("CREATE DATASETS AND LOADERS")

class SeqDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_ds = SeqDataset(X_train_scaled, y_train)
val_ds   = SeqDataset(X_val_scaled, y_val)
test_ds  = SeqDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

log(f"Number of train batches = {len(train_loader)}")
log(f"Number of val batches   = {len(val_loader)}")
log(f"Number of test batches  = {len(test_loader)}")


CREATE DATASETS AND LOADERS
[INFO] Number of train batches = 6
[INFO] Number of val batches   = 1
[INFO] Number of test batches  = 2


## Define the LSTM and GRU autoencoders

Both models share the same design:
- recurrent encoder
- latent projection
- repeated latent sequence
- recurrent decoder
- output projection back to the original feature dimension

In [8]:
banner("DEFINE AUTOENCODER MODELS")

class BaseSeqAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=32):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim

    def reconstruct(self, x):
        raise NotImplementedError

    def encode(self, x):
        raise NotImplementedError

    def forward(self, x):
        return self.reconstruct(x)

class LSTMAutoencoder(BaseSeqAutoencoder):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=32):
        super().__init__(input_dim, hidden_dim, latent_dim)
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        _, (h_n, _) = self.encoder(x)
        h = h_n[-1]
        z = self.to_latent(h)
        return z

    def reconstruct(self, x):
        seq_len = x.size(1)
        z = self.encode(x)
        dec_init = self.from_latent(z)
        dec_seq = dec_init.unsqueeze(1).repeat(1, seq_len, 1)
        dec_out, _ = self.decoder(dec_seq)
        out = self.output_layer(dec_out)
        return out

class GRUAutoencoder(BaseSeqAutoencoder):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=32):
        super().__init__(input_dim, hidden_dim, latent_dim)
        self.encoder = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        _, h_n = self.encoder(x)
        h = h_n[-1]
        z = self.to_latent(h)
        return z

    def reconstruct(self, x):
        seq_len = x.size(1)
        z = self.encode(x)
        dec_init = self.from_latent(z)
        dec_seq = dec_init.unsqueeze(1).repeat(1, seq_len, 1)
        dec_out, _ = self.decoder(dec_seq)
        out = self.output_layer(dec_out)
        return out

INPUT_DIM = X_train_scaled.shape[-1]
HIDDEN_DIM = 64
LATENT_DIM = 32

lstm_ae = LSTMAutoencoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM).to(DEVICE)
gru_ae  = GRUAutoencoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM).to(DEVICE)

log(f"Input dim = {INPUT_DIM}, hidden dim = {HIDDEN_DIM}, latent dim = {LATENT_DIM}")


DEFINE AUTOENCODER MODELS
[INFO] Input dim = 16, hidden dim = 64, latent dim = 32


## Training utilities

We use:
- MSE reconstruction loss
- Adam optimizer
- early stopping on validation loss

In [9]:
banner("DEFINE TRAINING UTILITIES")

def epoch_reconstruction_loss(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    if train_mode:
        model.train()
    else:
        model.eval()

    losses = []

    for batch in loader:
        if isinstance(batch, (list, tuple)):
            x = batch[0].to(DEVICE)
        else:
            x = batch.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        recon = model(x)
        loss = criterion(recon, x)

        if train_mode:
            loss.backward()
            optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses)) if losses else np.nan

def train_autoencoder(model, train_loader, val_loader, model_name, lr=1e-3, epochs=30, patience=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": []
    }

    best_val = float("inf")
    best_state = None
    best_epoch = -1
    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        banner(f"{model_name} — EPOCH {epoch}/{epochs}")

        train_loss = epoch_reconstruction_loss(model, train_loader, criterion, optimizer=optimizer)
        val_loss = epoch_reconstruction_loss(model, val_loader, criterion, optimizer=None)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        log(f"{model_name} | epoch={epoch} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            log(f"{model_name} early stopping triggered at epoch {epoch}", "WARN")
            break

    elapsed = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    return history, best_val, best_epoch, elapsed

def get_reconstruction_errors(model, X):
    model.eval()
    with torch.no_grad():
        x_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        recon = model(x_tensor)
        errors = ((recon - x_tensor) ** 2).mean(dim=(1, 2)).cpu().numpy()
    return errors

def get_latent_embeddings(model, X):
    model.eval()
    with torch.no_grad():
        x_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        z = model.encode(x_tensor).cpu().numpy()
    return z


DEFINE TRAINING UTILITIES


## Train the LSTM Autoencoder

In [10]:
banner("TRAIN LSTM AUTOENCODER")

lstm_history, lstm_best_val, lstm_best_epoch, lstm_train_time = timed_run(
    "train LSTM AE",
    train_autoencoder,
    lstm_ae, train_loader, val_loader, "LSTM Autoencoder", 1e-3, 30, 5
)

log(f"LSTM best val loss = {lstm_best_val:.6f}")
log(f"LSTM best epoch = {lstm_best_epoch}")
log(f"LSTM training time = {lstm_train_time:.2f} sec")


TRAIN LSTM AUTOENCODER
[INFO] START: train LSTM AE

LSTM Autoencoder — EPOCH 1/30
[INFO] LSTM Autoencoder | epoch=1 | train_loss=0.991008 | val_loss=0.487144

LSTM Autoencoder — EPOCH 2/30
[INFO] LSTM Autoencoder | epoch=2 | train_loss=0.938307 | val_loss=0.474737

LSTM Autoencoder — EPOCH 3/30
[INFO] LSTM Autoencoder | epoch=3 | train_loss=0.879101 | val_loss=0.446375

LSTM Autoencoder — EPOCH 4/30
[INFO] LSTM Autoencoder | epoch=4 | train_loss=0.807739 | val_loss=0.452546

LSTM Autoencoder — EPOCH 5/30
[INFO] LSTM Autoencoder | epoch=5 | train_loss=0.769172 | val_loss=0.432772

LSTM Autoencoder — EPOCH 6/30
[INFO] LSTM Autoencoder | epoch=6 | train_loss=0.755585 | val_loss=0.427697

LSTM Autoencoder — EPOCH 7/30
[INFO] LSTM Autoencoder | epoch=7 | train_loss=0.704684 | val_loss=0.427017

LSTM Autoencoder — EPOCH 8/30
[INFO] LSTM Autoencoder | epoch=8 | train_loss=0.715253 | val_loss=0.426629

LSTM Autoencoder — EPOCH 9/30
[INFO] LSTM Autoencoder | epoch=9 | train_loss=0.675652 | val

## Train the GRU Autoencoder

In [11]:
banner("TRAIN GRU AUTOENCODER")

gru_history, gru_best_val, gru_best_epoch, gru_train_time = timed_run(
    "train GRU AE",
    train_autoencoder,
    gru_ae, train_loader, val_loader, "GRU Autoencoder", 1e-3, 30, 5
)

log(f"GRU best val loss = {gru_best_val:.6f}")
log(f"GRU best epoch = {gru_best_epoch}")
log(f"GRU training time = {gru_train_time:.2f} sec")


TRAIN GRU AUTOENCODER
[INFO] START: train GRU AE

GRU Autoencoder — EPOCH 1/30
[INFO] GRU Autoencoder | epoch=1 | train_loss=0.969837 | val_loss=0.483261

GRU Autoencoder — EPOCH 2/30
[INFO] GRU Autoencoder | epoch=2 | train_loss=0.879537 | val_loss=0.441388

GRU Autoencoder — EPOCH 3/30
[INFO] GRU Autoencoder | epoch=3 | train_loss=0.796714 | val_loss=0.447176

GRU Autoencoder — EPOCH 4/30
[INFO] GRU Autoencoder | epoch=4 | train_loss=0.719684 | val_loss=0.426807

GRU Autoencoder — EPOCH 5/30
[INFO] GRU Autoencoder | epoch=5 | train_loss=0.694143 | val_loss=0.427947

GRU Autoencoder — EPOCH 6/30
[INFO] GRU Autoencoder | epoch=6 | train_loss=0.701792 | val_loss=0.423644

GRU Autoencoder — EPOCH 7/30
[INFO] GRU Autoencoder | epoch=7 | train_loss=0.678888 | val_loss=0.419912

GRU Autoencoder — EPOCH 8/30
[INFO] GRU Autoencoder | epoch=8 | train_loss=0.661366 | val_loss=0.415487

GRU Autoencoder — EPOCH 9/30
[INFO] GRU Autoencoder | epoch=9 | train_loss=0.654160 | val_loss=0.415411

GRU 

## Plot training and validation loss curves

This is one of the main presentation figures for the temporal comparison.

In [12]:
banner("PLOT LOSS CURVES")

loss_df = pd.concat([
    pd.DataFrame({
        "epoch": lstm_history["epoch"],
        "train_loss": lstm_history["train_loss"],
        "val_loss": lstm_history["val_loss"],
        "model": "LSTM"
    }),
    pd.DataFrame({
        "epoch": gru_history["epoch"],
        "train_loss": gru_history["train_loss"],
        "val_loss": gru_history["val_loss"],
        "model": "GRU"
    })
], ignore_index=True)

loss_long = loss_df.melt(
    id_vars=["epoch", "model"],
    value_vars=["train_loss", "val_loss"],
    var_name="loss_type",
    value_name="loss"
)

fig_loss = px.line(
    loss_long,
    x="epoch",
    y="loss",
    color="model",
    line_dash="loss_type",
    markers=True,
    title="LSTM vs GRU Autoencoder — Train/Validation Loss Curves"
)
fig_loss.update_layout(template="plotly_white")
fig_loss.show()

save_plot(fig_loss, "nb6_loss_curves")


PLOT LOSS CURVES


[WARN] Saved nb6_loss_curves.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Evaluate reconstruction loss on train / val / test

We compare:
- mean reconstruction error
- median reconstruction error
- train time

In [13]:
banner("RECONSTRUCTION ERROR EVALUATION")

# LSTM
lstm_train_err = get_reconstruction_errors(lstm_ae, X_train_scaled)
lstm_val_err   = get_reconstruction_errors(lstm_ae, X_val_scaled)
lstm_test_err  = get_reconstruction_errors(lstm_ae, X_test_scaled)

# GRU
gru_train_err = get_reconstruction_errors(gru_ae, X_train_scaled)
gru_val_err   = get_reconstruction_errors(gru_ae, X_val_scaled)
gru_test_err  = get_reconstruction_errors(gru_ae, X_test_scaled)

comparison_rows = [
    {
        "model": "LSTM",
        "best_val_loss": lstm_best_val,
        "best_epoch": lstm_best_epoch,
        "train_time_sec": lstm_train_time,
        "train_recon_mean": float(np.mean(lstm_train_err)),
        "val_recon_mean": float(np.mean(lstm_val_err)),
        "test_recon_mean": float(np.mean(lstm_test_err)),
        "train_recon_median": float(np.median(lstm_train_err)),
        "val_recon_median": float(np.median(lstm_val_err)),
        "test_recon_median": float(np.median(lstm_test_err)),
    },
    {
        "model": "GRU",
        "best_val_loss": gru_best_val,
        "best_epoch": gru_best_epoch,
        "train_time_sec": gru_train_time,
        "train_recon_mean": float(np.mean(gru_train_err)),
        "val_recon_mean": float(np.mean(gru_val_err)),
        "test_recon_mean": float(np.mean(gru_test_err)),
        "train_recon_median": float(np.median(gru_train_err)),
        "val_recon_median": float(np.median(gru_val_err)),
        "test_recon_median": float(np.median(gru_test_err)),
    },
]

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

comparison_df.to_csv("nb6_temporal_model_comparison.csv", index=False)
log("Saved nb6_temporal_model_comparison.csv")


RECONSTRUCTION ERROR EVALUATION


,model,best_val_loss,best_epoch,train_time_sec,train_recon_mean,val_recon_mean,test_recon_mean,train_recon_median,val_recon_median,test_recon_median
0,LSTM,0.401665,30,1.616108,0.594315,0.401665,0.508348,0.401276,0.395802,0.452853
1,GRU,0.390505,29,2.363033,0.572474,0.390505,0.454893,0.384876,0.388159,0.415707


[INFO] Saved nb6_temporal_model_comparison.csv


## Plot model comparison summary

This chart summarizes the core comparison metrics for the two temporal models.

In [14]:
banner("PLOT TEMPORAL MODEL COMPARISON")

plot_cols = ["best_val_loss", "test_recon_mean", "test_recon_median", "train_time_sec"]
comparison_long = comparison_df.melt(
    id_vars=["model"],
    value_vars=plot_cols,
    var_name="metric",
    value_name="value"
)

fig_cmp = px.bar(
    comparison_long,
    x="metric",
    y="value",
    color="model",
    barmode="group",
    title="Temporal Model Comparison Summary"
)
fig_cmp.update_layout(template="plotly_white")
fig_cmp.show()

save_plot(fig_cmp, "nb6_temporal_model_comparison")


PLOT TEMPORAL MODEL COMPARISON


[WARN] Saved nb6_temporal_model_comparison.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Plot reconstruction error distributions

These plots are useful to show how reconstruction error behaves across:
- train
- validation
- test
- model type

In [15]:
banner("PLOT RECONSTRUCTION ERROR DISTRIBUTIONS")

recon_df = pd.concat([
    pd.DataFrame({"error": lstm_train_err, "split": "train", "model": "LSTM", "label": y_train_raw}),
    pd.DataFrame({"error": lstm_val_err,   "split": "val",   "model": "LSTM", "label": y_val_raw}),
    pd.DataFrame({"error": lstm_test_err,  "split": "test",  "model": "LSTM", "label": y_test_raw}),
    pd.DataFrame({"error": gru_train_err,  "split": "train", "model": "GRU",  "label": y_train_raw}),
    pd.DataFrame({"error": gru_val_err,    "split": "val",   "model": "GRU",  "label": y_val_raw}),
    pd.DataFrame({"error": gru_test_err,   "split": "test",  "model": "GRU",  "label": y_test_raw}),
], ignore_index=True)

fig_recon = px.box(
    recon_df,
    x="split",
    y="error",
    color="model",
    points=False,
    title="Reconstruction Error by Split and Model"
)
fig_recon.update_layout(template="plotly_white")
fig_recon.show()

save_plot(fig_recon, "nb6_reconstruction_error_by_split")


PLOT RECONSTRUCTION ERROR DISTRIBUTIONS


[WARN] Saved nb6_reconstruction_error_by_split.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Reconstruction error by class on the test set

This shows whether one class is harder to reconstruct than the other.

In [16]:
banner("PLOT TEST RECONSTRUCTION ERROR BY CLASS")

recon_test_class_df = pd.concat([
    pd.DataFrame({"error": lstm_test_err, "label": y_test_raw, "model": "LSTM"}),
    pd.DataFrame({"error": gru_test_err,  "label": y_test_raw, "model": "GRU"}),
], ignore_index=True)

fig_recon_class = px.box(
    recon_test_class_df,
    x="label",
    y="error",
    color="model",
    points="all",
    title="Test Reconstruction Error by Class"
)
fig_recon_class.update_layout(template="plotly_white")
fig_recon_class.show()

save_plot(fig_recon_class, "nb6_test_reconstruction_error_by_class")


PLOT TEST RECONSTRUCTION ERROR BY CLASS


[WARN] Saved nb6_test_reconstruction_error_by_class.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Extract latent embeddings

We use the latent embeddings to evaluate:
- class separability
- visual structure in latent space

This prepares the transition toward the prototype notebook.

In [17]:
banner("EXTRACT LATENT EMBEDDINGS")

lstm_z_train = get_latent_embeddings(lstm_ae, X_train_scaled)
lstm_z_val   = get_latent_embeddings(lstm_ae, X_val_scaled)
lstm_z_test  = get_latent_embeddings(lstm_ae, X_test_scaled)

gru_z_train = get_latent_embeddings(gru_ae, X_train_scaled)
gru_z_val   = get_latent_embeddings(gru_ae, X_val_scaled)
gru_z_test  = get_latent_embeddings(gru_ae, X_test_scaled)

log(f"LSTM latent test shape = {lstm_z_test.shape}")
log(f"GRU latent test shape  = {gru_z_test.shape}")


EXTRACT LATENT EMBEDDINGS
[INFO] LSTM latent test shape = (68, 32)
[INFO] GRU latent test shape  = (68, 32)


## Compute latent separability

We use **silhouette score** on the test latent embeddings as a simple separability indicator.

In [18]:
banner("COMPUTE LATENT SEPARABILITY")

def safe_silhouette(Z, y):
    try:
        if len(np.unique(y)) < 2:
            return np.nan
        return float(silhouette_score(Z, y))
    except Exception:
        return np.nan

lstm_silhouette = safe_silhouette(lstm_z_test, y_test)
gru_silhouette  = safe_silhouette(gru_z_test, y_test)

comparison_df["test_latent_silhouette"] = [
    lstm_silhouette,
    gru_silhouette
]

display(comparison_df)
comparison_df.to_csv("nb6_temporal_model_comparison.csv", index=False)
log("Updated nb6_temporal_model_comparison.csv with silhouette score")


COMPUTE LATENT SEPARABILITY


,model,best_val_loss,best_epoch,train_time_sec,train_recon_mean,val_recon_mean,test_recon_mean,train_recon_median,val_recon_median,test_recon_median,test_latent_silhouette
0,LSTM,0.401665,30,1.616108,0.594315,0.401665,0.508348,0.401276,0.395802,0.452853,0.031914
1,GRU,0.390505,29,2.363033,0.572474,0.390505,0.454893,0.384876,0.388159,0.415707,0.013141


[INFO] Updated nb6_temporal_model_comparison.csv with silhouette score


## Plot latent-space visualizations with PCA

For presentation clarity, we project the latent embeddings to 2D using PCA.

In [19]:
banner("PLOT LATENT SPACE WITH PCA")

def pca_latent_df(Z, labels, model_name):
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    Z2 = pca.fit_transform(Z)
    df_plot = pd.DataFrame({
        "pc1": Z2[:, 0],
        "pc2": Z2[:, 1],
        "label": labels,
        "model": model_name
    })
    return df_plot

lstm_plot_df = pca_latent_df(lstm_z_test, y_test_raw, "LSTM")
gru_plot_df  = pca_latent_df(gru_z_test, y_test_raw, "GRU")

fig_lstm_latent = px.scatter(
    lstm_plot_df,
    x="pc1",
    y="pc2",
    color="label",
    title="LSTM Autoencoder — Test Latent Space (PCA)"
)
fig_lstm_latent.update_layout(template="plotly_white")
fig_lstm_latent.show()

fig_gru_latent = px.scatter(
    gru_plot_df,
    x="pc1",
    y="pc2",
    color="label",
    title="GRU Autoencoder — Test Latent Space (PCA)"
)
fig_gru_latent.update_layout(template="plotly_white")
fig_gru_latent.show()

save_plot(fig_lstm_latent, "nb6_lstm_latent_pca")
save_plot(fig_gru_latent, "nb6_gru_latent_pca")


PLOT LATENT SPACE WITH PCA


[WARN] Saved nb6_lstm_latent_pca.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)
[WARN] Saved nb6_gru_latent_pca.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Select the best temporal model

Selection rule:
1. lower validation reconstruction loss
2. if close, use lower test reconstruction mean
3. if still close, use higher test latent silhouette

In [20]:
banner("SELECT BEST TEMPORAL MODEL")

temp_rank = comparison_df.copy()
temp_rank = temp_rank.sort_values(
    by=["best_val_loss", "test_recon_mean", "test_latent_silhouette"],
    ascending=[True, True, False]
).reset_index(drop=True)

best_temporal_model = temp_rank.iloc[0]["model"]
log(f"Best temporal model selected = {best_temporal_model}")

display(temp_rank)


SELECT BEST TEMPORAL MODEL
[INFO] Best temporal model selected = GRU


,model,best_val_loss,best_epoch,train_time_sec,train_recon_mean,val_recon_mean,test_recon_mean,train_recon_median,val_recon_median,test_recon_median,test_latent_silhouette
0,GRU,0.390505,29,2.363033,0.572474,0.390505,0.454893,0.384876,0.388159,0.415707,0.013141
1,LSTM,0.401665,30,1.616108,0.594315,0.401665,0.508348,0.401276,0.395802,0.452853,0.031914


## Save model artifacts and summary

We save:
- model weights
- latent embeddings
- summaries
- presentation figures

In [21]:
banner("SAVE NOTEBOOK 6 OUTPUTS")

# Save model weights
torch.save(lstm_ae.state_dict(), "nb6_lstm_autoencoder.pt")
torch.save(gru_ae.state_dict(), "nb6_gru_autoencoder.pt")

# Save latent embeddings
np.savez_compressed(
    "nb6_lstm_latents_test.npz",
    Z=lstm_z_test,
    y=y_test_raw
)
np.savez_compressed(
    "nb6_gru_latents_test.npz",
    Z=gru_z_test,
    y=y_test_raw
)

summary = {
    "window_size": WINDOW_SIZE,
    "target_classes": class_names,
    "lstm_best_val_loss": float(lstm_best_val),
    "lstm_best_epoch": int(lstm_best_epoch),
    "lstm_train_time_sec": float(lstm_train_time),
    "gru_best_val_loss": float(gru_best_val),
    "gru_best_epoch": int(gru_best_epoch),
    "gru_train_time_sec": float(gru_train_time),
    "lstm_test_recon_mean": float(np.mean(lstm_test_err)),
    "gru_test_recon_mean": float(np.mean(gru_test_err)),
    "lstm_test_latent_silhouette": float(lstm_silhouette) if not np.isnan(lstm_silhouette) else None,
    "gru_test_latent_silhouette": float(gru_silhouette) if not np.isnan(gru_silhouette) else None,
    "best_temporal_model": best_temporal_model
}

with open("nb6_temporal_model_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

log("Saved Notebook 6 artifacts")


SAVE NOTEBOOK 6 OUTPUTS
[INFO] Saved Notebook 6 artifacts
